# Reading Why: interactive demo

Ask a small transformer whether a hypothesis follows from a premise, and see **why** it answers: the Born evidence
per word (BEW), exact, from the paper *Reading Why: Born-Measured Attention with Exact Word Evidence on Light and
Qubits*. Runs on a CPU; run the cells in order.

The task is natural language inference on e-SNLI: given a premise and a hypothesis, is the hypothesis
**entailed**, **neutral**, or a **contradiction**? Human annotators also highlighted the words that decide.

In [ ]:
#@title Setup (run this first)
# Get the code (in Colab: a fresh copy of the repository; when opened from a clone: that clone)
import os, sys
if os.path.exists("ui.py"):
    ROOT = ".."
elif os.path.exists("demo/ui.py"):
    ROOT = "."
elif os.path.exists("reading-why/.git"):
    !git -C reading-why pull -q        # a runtime that already has the code: bring it up to date
    ROOT = "reading-why"
else:
    !git clone -q --depth 1 https://github.com/stein-research/reading-why
    ROOT = "reading-why"
sys.path.insert(0, os.path.join(ROOT, "demo"))
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
import importlib, ui, reading_why as rw
importlib.reload(rw); importlib.reload(ui)      # the latest code, also when the runtime already had it
print("ready")

## Pick a pair, see why

Step through the test pairs with **◀ prev / next ▶** (the three labels in turn; pick one label to stay in it),
draw one with **random**, type a test index, or type your own premise and hypothesis and press **Explain**. Choose
the model with the toggle. You get:

- the class probabilities;
- the **evidence** of every word for the answer: red pushes towards it, blue against. For the Born Transformer
  the words' evidence adds up *exactly* to the class probability above its share b (about 1/3); for the softmax
  transformer it is ordinary integrated gradients, which does not add up;
- the **attention** each word receives, the quantity most often read as a model's reason;
- yellow underlines: the words the annotators highlighted.

Then pick one word or several under **pick words, then delete** and press **delete selected** to see whether the
answer changes (**clear** starts again). In the paper, deleting the word the Born Transformer's evidence ranks first
changes its answer in 45 % of test pairs, the word its attention ranks first in 32 %; deleting the five words its
evidence ranks last changes it in 1 %. Each pair takes a few seconds.

In [ ]:
#@title The demo
ui.demo()

## Without widgets

The same in plain code, for a pair of your own.

In [ ]:
r = rw.explain(ui.model("born"), "A man is playing a guitar on stage.", "A man is performing music.")
print(r["answer"], r["probs"].round(3))
for w, e, a in zip(r["words"], r["evidence"], r["attention"]):
    print(f"{w:>12s}  evidence {e:+.4f}   attention {a:.3f}")
print(f"sum of evidence {r['evidence'].sum():+.6f} = P - b = {r['explained']:+.6f}")